In [1]:
!pip install scikit-learn joblib mistralai beautifulsoup4 pypdf requests

   ---------------------------------------- 0.0/331.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/331.4 kB ? eta -:--:--
   - -------------------------------------- 10.2/331.4 kB ? eta -:--:--
   -------- ------------------------------ 71.7/331.4 kB 653.6 kB/s eta 0:00:01
   --------------------------- ------------ 225.3/331.4 kB 1.5 MB/s eta 0:00:01
   ---------------------------------------  327.7/331.4 kB 1.8 MB/s eta 0:00:01
   ---------------------------------------- 331.4/331.4 kB 1.7 MB/s eta 0:00:00


In [2]:
import io
import logging
import requests
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
from pypdf import PdfReader

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
from mistralai import Mistral

logging.getLogger("pypdf").setLevel(logging.ERROR)

In [3]:
def same_domain(url, base_hostnames):
    try:
        netloc = urlparse(url).netloc.lower()
        for h in base_hostnames:
            if h in netloc:
                return True
        return False
    except:
        return False

def crawl_for_pdfs(start_urls, base_hostnames, max_docs=450, max_pages=600):
    to_visit = list(start_urls)
    visited = set()
    pdf_links = set()
    while to_visit and len(visited) < max_pages and len(pdf_links) < max_docs:
        url = to_visit.pop(0)
        if url in visited:
            continue
        visited.add(url)
        try:
            r = requests.get(url, timeout=15)
            ct = r.headers.get("Content-Type", "").lower()
            if "pdf" in ct:
                pdf_links.add(url)
                continue
            soup = BeautifulSoup(r.text, "html.parser")
            for a in soup.find_all("a", href=True):
                href = a["href"]
                full = href if href.startswith("http") else urljoin(url, href)
                if full.lower().endswith(".pdf") or ".pdf?" in full.lower():
                    pdf_links.add(full)
                else:
                    if same_domain(full, base_hostnames) and full not in visited and len(to_visit) < max_pages:
                        to_visit.append(full)
            if len(pdf_links) >= max_docs:
                break
        except:
            pass
    return list(pdf_links)

def download_and_extract(url, min_chars=500):
    r = requests.get(url, timeout=25)
    ct = r.headers.get("Content-Type", "").lower()
    if "pdf" in ct or url.lower().endswith(".pdf"):
        try:
            reader = PdfReader(io.BytesIO(r.content))
            pages = []
            for p in reader.pages:
                t = p.extract_text() or ""
                pages.append(t)
            text = "\n".join(pages)
            if len(text.strip()) < min_chars:
                raise ValueError("too short")
            return text
        except:
            soup = BeautifulSoup(r.text, "html.parser")
            return soup.get_text(" ", strip=True)
    else:
        soup = BeautifulSoup(r.text, "html.parser")
        return soup.get_text(" ", strip=True)

def build_corpus_from_urls(urls, min_chars=800):
    texts = []
    used_urls = []
    for u in urls:
        try:
            t = download_and_extract(u, min_chars=min_chars)
            if len(t) >= min_chars:
                texts.append(t)
                used_urls.append(u)
        except:
            pass
    return texts, used_urls

In [4]:
start_urls = [
    "https://www.nice.org.uk/guidance/published?type=guideline",
    "https://www.has-sante.fr/jcms/r_1500918/fr/recommandations-professionnelles",
    "https://www.sign.ac.uk/our-guidelines/"
]

base_hostnames = [
    "nice.org.uk",
    "has-sante.fr",
    "sign.ac.uk"
]

pdf_links = crawl_for_pdfs(start_urls, base_hostnames, max_docs=500, max_pages=800)
len(pdf_links)

417

In [5]:
documents, doc_urls = build_corpus_from_urls(pdf_links, min_chars=1000)
len(documents), len(doc_urls)

(414, 414)

In [6]:
vectorizer_path = "tfidf_vectorizer.joblib"
matrix_path = "document_matrix.joblib"

In [7]:
def index_document(documents: list[str]):
    vectorizer = TfidfVectorizer(max_features=50000)
    X = vectorizer.fit_transform(documents)
    joblib.dump(vectorizer, vectorizer_path)
    joblib.dump(X, matrix_path)

def query_indices(documents: list[str], query: str, top_n: int):
    vectorizer = joblib.load(vectorizer_path)
    X = joblib.load(matrix_path)
    q_vec = vectorizer.transform([query])
    sims = (X @ q_vec.T).toarray().ravel()
    idx = np.argsort(-sims)[:top_n]
    return idx

def rag_answer_with_docs(documents: list[str], urls: list[str], query: str, top_n: int = 3):
    idx = query_indices(documents, query, top_n)
    retrieved_docs = [documents[i] for i in idx]
    retrieved_urls = [urls[i] for i in idx]
    context = ""
    for i, d in enumerate(retrieved_docs, 1):
        context += f"Document {i}:\n{d[:4000]}\n\n"
    prompt = (
        "You are a helpful assistant.\n"
        "Use only the information from the clinical guideline documents below and your basic understanding of language.\n"
        "If something is not covered, say that the documents do not provide enough information.\n\n"
        f"{context}"
        f"User question: {query}\n\n"
        "Answer clearly and concisely."
    )
    client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])
    res = client.chat.complete(
        model="mistral-small-latest",
        messages=[
            {"role": "system", "content": "You answer using the provided documents as main source of truth."},
            {"role": "user", "content": prompt}
        ]
    )
    answer = res.choices[0].message.content
    return retrieved_docs, retrieved_urls, answer

index_document(documents)

In [ ]:
import os
os.environ["MISTRAL_API_KEY"] = "XXXXXXXXXXXXXXXXXXXXXXXXXXX"

In [9]:
questions = [
    "What is the general purpose of clinical guidelines according to these documents?",
    "What kind of information do these guidelines usually give to clinicians?",
    "How do the guidelines suggest improving consistency or quality of care?",
    "What caveats or warnings do the guidelines give about how they should be used?"
]

for q in questions:
    print("QUESTION:", q)
    retrieved_docs, retrieved_urls, answer = rag_answer_with_docs(documents, doc_urls, q, top_n=3)
    print("\nTop 3 document URLs:")
    for i, u in enumerate(retrieved_urls, 1):
        print("Document", i, "URL:", u)
    print("\nANSWER:")
    print(answer)
    print("\n" + "-" * 80 + "\n")

QUESTION: What is the general purpose of clinical guidelines according to these documents?

Top 3 document URLs:
Document 1 URL: https://www.sign.ac.uk/media/1642/sign50_2011.pdf
Document 2 URL: https://www.sign.ac.uk/media/2329/sign50_2019.pdf
Document 3 URL: https://www.sign.ac.uk/media/2038/sign50_2019.pdf

ANSWER:
The documents do not explicitly state the general purpose of clinical guidelines. They focus more on the development, updating, and implementation of guidelines rather than their overall purpose. Therefore, based on the provided documents, I cannot provide a specific answer to the user's question.

--------------------------------------------------------------------------------

QUESTION: What kind of information do these guidelines usually give to clinicians?

Top 3 document URLs:
Document 1 URL: https://www.sign.ac.uk/media/2333/20252901-sign-100-v10.pdf
Document 2 URL: https://www.sign.ac.uk/media/1558/sign_copyright_request.pdf
Document 3 URL: https://www.sign.ac.uk/m